In [25]:
# Этап 1. Загрузка и предобработка данных
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import numpy as np

# Устройство
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Трансформации
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Загрузка
train_dir = '/kaggle/input/datasets/viacheslavr/for-project-2/ogyeiv2/train'
test_dir = '/kaggle/input/datasets/viacheslavr/for-project-2/ogyeiv2/test'

train_dataset = ImageFolder(train_dir, transform=train_transforms)
test_dataset = ImageFolder(test_dir, transform=test_transforms)

# DataLoader
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Вывод
print(f"Классов: {len(train_dataset.classes)}")
print(f"Train: {len(train_dataset)} изображений")
print(f"Test: {len(test_dataset)} изображений")

Device: cuda
Классов: 84
Train: 2352 изображений
Test: 504 изображений


In [26]:
# Этап 2. Объявление модели
from torchvision.models import resnet50
import torch.nn as nn

model = resnet50(weights='IMAGENET1K_V2')

# Заморозка
for param in model.parameters():
    param.requires_grad = False

# Новый классификатор
model.fc = nn.Linear(2048, len(train_dataset.classes))
model = model.to(device)

print(f"Модель создана. Классов: {len(train_dataset.classes)}")

Модель создана. Классов: 84


In [27]:
# Этап 3. Обучение
import torch.optim as optim
from tqdm import tqdm

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

best_acc = 0

for epoch in range(10):
    # Обучение
    model.train()
    train_loss, train_correct = 0, 0
    for x, y in tqdm(train_loader, desc=f'Epoch {epoch+1}'):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        train_correct += (out.argmax(1) == y).sum().item()
    
    train_acc = 100 * train_correct / len(train_dataset)
    
    # Тест
    model.eval()
    test_correct = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            test_correct += (out.argmax(1) == y).sum().item()
    
    test_acc = 100 * test_correct / len(test_dataset)
    test_errors = len(test_dataset) - test_correct
    
    print(f"\nЭпоха {epoch+1}:")
    print(f"  Train Loss: {train_loss/len(train_loader):.4f}, Train Acc: {train_acc:.2f}%")
    print(f"  Test Acc: {test_acc:.2f}%, Ошибок: {test_errors}/{len(test_dataset)}")
    
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), 'meds_classifier.pt')
        print(f"  ✓ Модель сохранена (Acc: {test_acc:.2f}%)")

print(f"\nЛучшая точность: {best_acc:.2f}%")

Epoch 1: 100%|██████████| 37/37 [04:25<00:00,  7.17s/it]



Эпоха 1:
  Train Loss: 4.2192, Train Acc: 11.56%
  Test Acc: 30.16%, Ошибок: 352/504
  ✓ Модель сохранена (Acc: 30.16%)


Epoch 2: 100%|██████████| 37/37 [04:15<00:00,  6.92s/it]



Эпоха 2:
  Train Loss: 3.5662, Train Acc: 56.38%
  Test Acc: 59.92%, Ошибок: 202/504
  ✓ Модель сохранена (Acc: 59.92%)


Epoch 3: 100%|██████████| 37/37 [04:17<00:00,  6.95s/it]


...


Эпоха 8:
  Train Loss: 1.5163, Train Acc: 93.54%
  Test Acc: 78.57%, Ошибок: 108/504


Epoch 9: 100%|██████████| 37/37 [04:11<00:00,  6.79s/it]



Эпоха 9:
  Train Loss: 1.3196, Train Acc: 93.71%
  Test Acc: 81.94%, Ошибок: 91/504
  ✓ Модель сохранена (Acc: 81.94%)


Epoch 10: 100%|██████████| 37/37 [04:11<00:00,  6.80s/it]



Эпоха 10:
  Train Loss: 1.1814, Train Acc: 95.37%
  Test Acc: 83.13%, Ошибок: 85/504
  ✓ Модель сохранена (Acc: 83.13%)

Лучшая точность: 83.13%


In [28]:
# Этап 4. Оценка качества
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# Загрузка модели
model.load_state_dict(torch.load('meds_classifier.pt'))
model.eval()

# Предсказания
all_preds, all_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        preds = model(x).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y.numpy())

# Метрики
accuracy = accuracy_score(all_labels, all_preds)
print(f"\n{'='*50}")
print(f"ОБЩАЯ ТОЧНОСТЬ: {accuracy*100:.2f}%")
print(f"{'='*50}\n")

# Метрики по классам
report = classification_report(all_labels, all_preds, 
                              target_names=train_dataset.classes,
                              output_dict=True)
df = pd.DataFrame(report).transpose()

# 5 худших
worst = df.drop(['accuracy', 'macro avg', 'weighted avg']).nsmallest(5, 'f1-score')

print("5 КЛАССОВ С ЧАСТЫМИ ОШИБКАМИ:")
for name, row in worst.iterrows():
    print(f"  {name}: F1={row['f1-score']:.3f}")

# Без ошибок
perfect = df[df['f1-score'] == 1.0]
print(f"\nКЛАССЫ БЕЗ ОШИБОК: {len(perfect)}")
for name in perfect.index[:5]:
    print(f"  {name}")

# Ответы
print(f"\n{'='*50}")
print("АНАЛИЗ")
print(f"{'='*50}")

print("\n1. ПОЧЕМУ ОШИБКИ НА 5 КЛАССАХ?")
print("   - Визуальное сходство таблеток")
print("   - Мало обучающих примеров")

print("\n2. ПОЧЕМУ КЛАССЫ БЕЗ ОШИБОК?")
print("   - Уникальный внешний вид")
print("   - Достаточно данных")

print("\n3. КАК УЛУЧШИТЬ?")
print("   - Разморозить больше слоев")
print("   - Увеличить аугментацию")
print("   - Использовать ансамбль")

print("\n4. ДОП. АНАЛИЗ:")
print("   - Grad-CAM визуализация")
print("   - Анализ по форме/цвету")


ОБЩАЯ ТОЧНОСТЬ: 83.13%

5 КЛАССОВ С ЧАСТЫМИ ОШИБКАМИ:
  meridian: F1=0.400
  teva_ambrobene_30_mg: F1=0.444
  covercard_plus_10_mg_2_5_mg_5_mg: F1=0.500
  jutavit_cink: F1=0.500
  sedatif_pc: F1=0.500

КЛАССЫ БЕЗ ОШИБОК: 22
  advil_ultra_forte
  algoflex_forte_dolo_400_mg
  algoflex_rapid_400_mg
  apranax_550_mg
  c_vitamin_teva_500_mg

АНАЛИЗ

1. ПОЧЕМУ ОШИБКИ НА 5 КЛАССАХ?
   - Визуальное сходство таблеток
   - Мало обучающих примеров

2. ПОЧЕМУ КЛАССЫ БЕЗ ОШИБОК?
   - Уникальный внешний вид
   - Достаточно данных

3. КАК УЛУЧШИТЬ?
   - Разморозить больше слоев
   - Увеличить аугментацию
   - Использовать ансамбль

4. ДОП. АНАЛИЗ:
   - Grad-CAM визуализация
   - Анализ по форме/цвету
